# Google Scholar Explorer

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook searches Google Scholar and exports structured citation data for literature reviews and bibliometric research. Enter search terms, set year filters, and download publication metadata — titles, authors, publication years, venues, citation counts, and URLs — as CSV or Excel.

The notebook uses the `scholarly` library to query Google Scholar directly — no API key required. Results are returned with full bibliographic metadata for integration with reference managers, qualitative analysis software, and bibliometric tools.

## Key Features

1. **No API Key Required**: Queries Google Scholar directly using the open-source `scholarly` library
2. **Keyword Search**: Search across all of Google Scholar's indexed publications
3. **Year Filtering**: Restrict results to a specific publication year range
4. **Configurable Result Count**: Fetch 10 to 200 results per search
5. **Checkpoint & Combine**: Large searches save progress in batches, then merge into one export
6. **Structured Export**: Download results as CSV or styled Excel for further analysis

## Workflow

1. **Configure**: Enter search terms, set year range and result count
2. **Fetch**: Retrieve publication data from Google Scholar
3. **Review**: Preview results in a table
4. **Export**: Download structured citation data for further analysis

## How Results Work

Google Scholar returns results in pages of 10. The `scholarly` library handles pagination automatically, but larger searches take longer (roughly 1–2 seconds per 10 results). The notebook saves checkpoint files every 50 results so that progress is preserved even if a search is interrupted.

**Rate limiting**: Google Scholar may temporarily block requests if too many are made in a short period. The notebook includes built-in delays between pages. If you encounter blocks, wait a few minutes and try again, or reduce your result count.

## Applications

This notebook supports literature reviews, bibliometric analysis, and citation mapping — identifying key publications in a field, tracking citation patterns over time, building structured reading lists, and discovering the scholarly landscape around research topics. The structured exports integrate with reference managers and qualitative analysis tools.

## Methodological Positioning

This notebook represents a **computational anthropology** approach — using programmatic data retrieval to support systematic literature engagement. The notebook collects and structures citation data but does not interpret it. Critical evaluation of sources remains the work of the researcher.

**Important**: Google Scholar indexes a broad range of sources with varying quality. Not all indexed items are peer-reviewed. Use disciplinary judgment when evaluating results.

## Target Audience

Designed for anthropologists and qualitative researchers conducting literature reviews — from graduate students mapping a dissertation field to research teams conducting systematic reviews or bibliometric analyses.

## Technical Approach

The notebook uses the **scholarly** library to query Google Scholar. It supports keyword search, year-range filtering, and automatic pagination. Results include title, authors, year, venue, abstract, citation count, and URLs. All processing runs locally in the notebook.

## License & Attribution

This notebook is released under the **CC BY-NC 4.0** license. You may adapt and share it for non-commercial purposes with proper attribution.

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Human Organization. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

## Setup and Installation

Install required Python packages and import libraries for Google Scholar retrieval, data processing, and interactive configuration. Run this cell first to ensure all dependencies are available.

In [ ]:
# Install required packages
!pip install scholarly pandas openpyxl ipywidgets -q

import os
import re
import unicodedata
import pandas as pd
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

from scholarly import scholarly
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def normalize_text(text):
    """Normalize unicode characters to ASCII-friendly equivalents."""
    if not isinstance(text, str):
        return text
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\u2011', '-').replace('\u2013', '-').replace('\u2014', '-')
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = text.replace('\u2026', '...')
    return text


def make_slug(query):
    """Create a clean filename slug from a query string."""
    slug = re.sub(r'[^a-zA-Z0-9]+', '_', query).strip('_')[:30]
    return slug if slug else 'search'


def style_excel(filepath):
    """Apply branded styling to an Excel file."""
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    header_fill = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    thin_border = Border(
        left=Side(style='thin', color='E7ECEF'), right=Side(style='thin', color='E7ECEF'),
        top=Side(style='thin', color='E7ECEF'), bottom=Side(style='thin', color='E7ECEF'),
    )
    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = thin_border
        for col in ws.columns:
            max_len = max(len(str(c.value or '')) for c in col)
            ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical='top', wrap_text=True)
                cell.border = thin_border
    wb.save(filepath)


print("\u2713 All packages installed and libraries loaded successfully")
print(f"\u2713 Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print("\U0001f4da Ready to configure your Google Scholar search!")

## Search Configuration

Configure your Google Scholar search using the interactive controls below. Set your search terms, year range, result count, and export format.

In [ ]:
# Interactive Search Interface

style = {'description_width': '120px'}
layout = widgets.Layout(width='500px')

instructions_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
instructions_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f4da Google Scholar Explorer</h2>'
instructions_html += '<p><strong>Welcome!</strong> This notebook searches Google Scholar and exports structured citation data for literature reviews.</p>'
instructions_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
instructions_html += '<ol>'
instructions_html += '<li><strong>Search:</strong> Enter keywords for your literature search</li>'
instructions_html += '<li><strong>Filter:</strong> Set year range and result count</li>'
instructions_html += '<li><strong>Fetch:</strong> Click the button to retrieve results (larger searches take longer)</li>'
instructions_html += '<li><strong>Export:</strong> Download as CSV or Excel</li>'
instructions_html += '</ol>'
instructions_html += '<p style="margin-top: 10px; color: #8B8C89; font-size: 0.9em;">'
instructions_html += 'Results are fetched in pages of 10 (~1\u20132 sec per page). Progress is saved every 50 results. '
instructions_html += 'If Google blocks requests temporarily, wait a few minutes and try again.</p>'
instructions_html += '</div>'

instructions = widgets.HTML(value=instructions_html)

search_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f50d Search</h3>')

query_input = widgets.Text(
    value='',
    placeholder='Enter search terms (e.g., "digital ethnography methods")',
    description='Keywords:',
    layout=layout,
    style=style
)

filter_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c5 Filters</h3>')

year_start_input = widgets.IntText(
    value=2000,
    description='From year:',
    style=style
)

year_end_input = widgets.IntText(
    value=2026,
    description='To year:',
    style=style
)

year_help = widgets.HTML(
    '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
    'Set both to 0 to search all years. Google Scholar year filtering is approximate.</p>'
)

max_results_input = widgets.IntSlider(
    value=50, min=10, max=200, step=10,
    description='Max results:',
    style=style
)

max_help = widgets.HTML(
    '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
    'Larger searches take longer (~1\u20132 sec per 10 results). Progress checkpoints save every 50 results.</p>'
)

export_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c1 Export</h3>')

format_input = widgets.Dropdown(
    options=[('CSV (.csv)', 'csv'), ('Excel (.xlsx)', 'xlsx')],
    value='csv',
    description='Format:',
    layout=layout,
    style=style
)

fetch_btn = widgets.Button(
    description='Search Scholar',
    button_style='',
    layout=widgets.Layout(width='200px', margin='20px 0'),
    style={'button_color': '#6096BA'}
)

out = widgets.Output()


def on_fetch_clicked(_):
    out.clear_output()
    with out:
        query = query_input.value.strip()
        if not query:
            print("\u26a0\ufe0f Please enter search terms.")
            return

        max_count = max_results_input.value
        year_lo = year_start_input.value if year_start_input.value > 0 else None
        year_hi = year_end_input.value if year_end_input.value > 0 else None

        print(f"\U0001f50d Searching Google Scholar for: {query}")
        if year_lo and year_hi:
            print(f"   Year range: {year_lo}\u2013{year_hi}")
        print(f"   Max results: {max_count}")
        print()

        try:
            search_kwargs = {}
            if year_lo:
                search_kwargs['year_low'] = year_lo
            if year_hi:
                search_kwargs['year_high'] = year_hi

            results_iter = scholarly.search_pubs(query, **search_kwargs)

            all_rows = []
            slug = make_slug(query)
            checkpoint_size = 50

            for i in range(max_count):
                try:
                    r = next(results_iter)
                except StopIteration:
                    print(f"   No more results (found {len(all_rows)} total)")
                    break

                bib = r.get('bib', {})
                authors = bib.get('author', [])
                if isinstance(authors, list):
                    authors = '; '.join(authors)

                all_rows.append({
                    'title': normalize_text(bib.get('title', '')),
                    'authors': normalize_text(authors),
                    'year': bib.get('pub_year', ''),
                    'venue': normalize_text(bib.get('venue', '')),
                    'abstract': normalize_text(bib.get('abstract', '')),
                    'citations': r.get('num_citations', 0),
                    'url': r.get('pub_url', r.get('eprint_url', '')),
                })

                # Progress
                count = len(all_rows)
                print(f"   \u2713 {count}/{max_count} results fetched...", end='\r')


            print()  # clear progress line

            if not all_rows:
                print("\u26a0\ufe0f No results found. Try different search terms or a broader year range.")
                return

            print()
            print(f"\u2713 Retrieved {len(all_rows)} publications")
            print()

            df = pd.DataFrame(all_rows)

            print("\U0001f4ca Preview:")
            display(df[['title', 'authors', 'year', 'citations']].head(15))

            # Export
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            base = f'google_scholar_{slug}_{timestamp}'
            fmt = format_input.value

            if fmt == 'xlsx':
                filepath = f'{base}.xlsx'
                df.to_excel(filepath, sheet_name='Scholar Results', index=False, engine='openpyxl')
                style_excel(filepath)
            else:
                filepath = f'{base}.csv'
                df.to_csv(filepath, index=False)

            print()
            print(f"\u2713 Saved: {filepath} ({len(df)} publications)")

            if IN_COLAB:
                colab_files.download(filepath)
                print(f"\U0001f4e5 Downloaded: {filepath}")


        except Exception as e:
            print(f"\u274c Error: {type(e).__name__}: {e}")
            if "429" in str(e) or "blocked" in str(e).lower() or "captcha" in str(e).lower():
                print("\n\U0001f6a7 Google Scholar may be rate-limiting requests.")
                print("   Wait a few minutes and try again, or reduce your result count.")


fetch_btn.on_click(on_fetch_clicked)

display(widgets.VBox([
    instructions,
    search_header,
    query_input,
    filter_header,
    widgets.HBox([year_start_input, year_end_input]),
    year_help,
    max_results_input,
    max_help,
    export_header,
    format_input,
    fetch_btn,
    out,
]))